In [10]:
from pathlib import Path 
import hashlib
import shutil 

In [11]:
PROJECT_ROOT = Path.cwd()

RAW_ROOT = PROJECT_ROOT / "pill-dataset"
IMAGE_ROOT = RAW_ROOT / "images"
LABEL_ROOT = RAW_ROOT / "labels"
OUTPUT_ROOT = PROJECT_ROOT / "pill-dataset-clean"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [12]:
def sha256_file(path: Path) -> str:
    """คำนวณ SHA-256 แบบทีละ Chunk เพื่อประหยัด RAM"""
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def clean_label(source: Path, destination: Path) -> None:
    """ทำความสะอาดไฟล์ label YOLO และ Clip พิกัดให้อยู่ในกรอบ 0.0 - 1.0"""
    text = source.read_text(encoding="utf-8").strip()
    if not text:
        destination.write_text("", encoding="utf-8")
        return

    output_lines = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue

        try:
            class_id = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            continue

        # แปลงจาก (center_x, center_y, w, h) -> (x1, y1, x2, y2)
        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2

        # ตัดขอบให้อยู่ในช่วง 0.0 - 1.0
        x1 = max(0.0, min(1.0, x1))
        y1 = max(0.0, min(1.0, y1))
        x2 = max(0.0, min(1.0, x2))
        y2 = max(0.0, min(1.0, y2))

        # ตรวจสอบขนาดกล่องหลัง Clip หากแบนหรือพิกัดกลับด้านให้ทิ้ง
        if x2 <= x1 or y2 <= y1:
            continue

        # แปลงกลับเป็นฟอร์แมต YOLO
        new_x = (x1 + x2) / 2
        new_y = (y1 + y2) / 2
        new_w = x2 - x1
        new_h = y2 - y1

        output_lines.append(
            f"{class_id} {new_x:.6f} {new_y:.6f} {new_w:.6f} {new_h:.6f}"
        )

    destination.write_text("\n".join(output_lines), encoding="utf-8")

In [13]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for split in ["train", "val"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print(f"สร้างโครงสร้างโฟลเดอร์เรียบร้อยที่: {OUTPUT_ROOT}")

สร้างโครงสร้างโฟลเดอร์เรียบร้อยที่: c:\Project\pill-object-detection\pill-dataset-clean


In [14]:
train_hashes = set()
train_image_dir = IMAGE_ROOT / "train"

train_images = [
    p for p in train_image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS
]

for image_path in train_images:
    train_hashes.add(sha256_file(image_path))

print(f"เก็บ Hash รูปภาพจาก Train สำเร็จทั้งหมด: {len(train_hashes)} รายการ")

เก็บ Hash รูปภาพจาก Train สำเร็จทั้งหมด: 10557 รายการ


In [15]:
stats = {
    "train": 0,
    "val": 0,
    "duplicates_removed": 0,
}

for split in ["train", "val"]:
    source_images = IMAGE_ROOT / split
    source_labels = LABEL_ROOT / split
    output_images = OUTPUT_ROOT / "images" / split
    output_labels = OUTPUT_ROOT / "labels" / split

    for image_path in source_images.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        # ตรวจสอบและตัดภาพใน val ที่ซ้ำกับ train ออก
        if split == "val":
            image_hash = sha256_file(image_path)
            if image_hash in train_hashes:
                stats["duplicates_removed"] += 1
                continue

        # คัดลอกรูปภาพ
        shutil.copy2(image_path, output_images / image_path.name)

        # จัดการไฟล์ Label
        label_path = source_labels / f"{image_path.stem}.txt"
        destination_label = output_labels / f"{image_path.stem}.txt"

        if label_path.exists():
            clean_label(label_path, destination_label)
        else:
            destination_label.write_text("", encoding="utf-8")

        stats[split] += 1

print("ประมวลผลเสร็จสิ้น:")
for k, v in stats.items():
    print(f"- {k}: {v}")

ประมวลผลเสร็จสิ้น:
- train: 10559
- val: 1506
- duplicates_removed: 2
